# Implementation of the Kramers-Wannier Duality on a Quantum Computer
**Author:** Arif  
**Output Configuration:** PDF Document (`mathtools`, `braket` enabled)

***

We verify the Kramers-Wannier duality on the Aer simulator. We first create a random ising state, using the technology used in the VQE initial state ansatz. Then we apply our $\mathcal{D}$ operator on it following fig.(3) of the paper. In another circuit we apply this state to ising lattice. We verify the duality in the sense of eq.(38) of the paper, 

\begin{align}
\text{Tr}_{\hat{\mathcal{H}}} \left( \hat{Z}_1 \hat{Z}_2 \mathcal{D}(\rho) \right) = \frac{1}{4} \text{Tr}_{\mathcal{H}} \left( X_1 P \rho \right)
\end{align}

Here the left and right hand side are computed postprocessing the data from the simulator run. We get the ratio of the two traces close to 0.25 as it should be. Both quantum circuits evaluate the exact same state instance to maintain verification identity conditions.

### 1-Qubit Basis Transformation Proof for $X$-Basis Expectation

We apply $Z_1 Z_2$ on the domain wall state, thus we need to compute the expectation value of $X_1$ in the Ising state. Since the quantum computer measures in the computational ($Z$) basis only, we rotate the state into the $X$ basis to extract $\langle X \rangle$ from the data. 

The trick is to apply the Hadamard gate $H$ on the corresponding Ising qubit prior to measurement. Then, $\langle X \rangle$ is equal to the probability of getting $0$ minus the probability of getting $1$. 

#### Derivation:
Let the state of our qubit be:
\begin{align}
\ket{\psi} = \alpha \ket{0} + \beta \ket{1}
\end{align}

The analytical expectation value of the Pauli-$X$ operator on this state evaluates to:
\begin{align}
\bra{\psi} X \ket{\psi} = \alpha^* \beta + \beta^* \alpha
\end{align}

Now, applying the Hadamard gate transformations ($H\ket{0} = \frac{\ket{0}+\ket{1}}{\sqrt{2}}$ and $H\ket{1} = \frac{\ket{0}-\ket{1}}{\sqrt{2}}$) yields the rotated state:
\begin{align}
H\ket{\psi} = \frac{\alpha + \beta}{\sqrt{2}} \ket{0} + \frac{\alpha - \beta}{\sqrt{2}} \ket{1}
\end{align}

When measuring this rotated state in the computational basis, the respective outcome probabilities are:
* Probability of measuring state $\ket{0}$:  
  \begin{align} P_0 = \frac{|\alpha + \beta|^2}{2} = \frac{(\alpha + \beta)(\alpha^* + \beta^*)}{2} = \frac{|\alpha|^2 + |\beta|^2 + \alpha\beta^* + \alpha^*\beta}{2} \end{align}
* Probability of measuring state $\ket{1}$:  
  \begin{align} P_1 = \frac{|\alpha - \beta|^2}{2} = \frac{(\alpha - \beta)(\alpha^* - \beta^*)}{2} = \frac{|\alpha|^2 + |\beta|^2 - \alpha\beta^* - \alpha^*\beta}{2} \end{align}

Subtracting the two outcome probabilities cancels the diagonal density matrix elements, leaving the cross-terms:
\begin{align}
P_0 - P_1 = \alpha^* \beta + \beta^* \alpha = \bra{\psi} X \ket{\psi}
\end{align}

Therefore, the expectation value $\langle X_1 \rangle$ is directly calculated during post-processing by computing $P_0 - P_1$ from the filtered code runs.


### Code Organization and Execution Logic

The verification is structured as a single, sequential workflow divided into four primary logical stages:

1. **State Preparation & Core Operator Synthesis**
   * **Initial State Blueprint**: Generates a random parameterized initial state $\rho$ on an $n$-qubit lattice using a standard `TwoLocal` variational circuit structure (VQE ansatz configuration). 
   * **Duality Operator Gate ($\mathcal{D}$)**: Explicitly constructs the dual mapping circuit layer by sequencing tracking interactions (the boundary mapping operator curly-$C$ and site field mapping operator curly-$S$).
   * **Charge Symmetry Projections ($P$)**: Builds custom tensor projection matrices to filter state populations inside specific charge symmetry configurations using dedicated auxiliary registers.
     

2. **Circuit A: Left-Hand Side (LHS) Duality Mapping**
   * Prepares the initial parameter-bound state on the main system lattice.
   * Enforces charge projection tracking constraints on both the base system and dual registers.
   * Applies the composite duality gate to transition into the domain wall basis.
   * Maps explicit bit-by-bit classical register allocations to read out the joint state strings securely without shifting coordinate keys.


3. **Circuit B: Right-Hand Side (RHS) Baseline Tracking**
   * Initializes the exact same random parameterized variational state on a standalone system to guarantee verification identity conditions.
   * Applies a single charge sector projection gate tracking line.
   * Rotates the target coordinate qubit into the $X$-measurement basis via a Hadamard gate, recording metrics with a uniform `measure_all()` capture call.


4. **Post-Processing & Identity Evaluation**
   * **LHS Evaluation**: Iterates over Circuit A's string outputs, post-selecting data matches where the 4-bit projection block equals `'0000'`, then aggregates expectation values according to the target domain wall parity signs ($Z_1 Z_2$).
   * **RHS Evaluation**: Iterates over Circuit B's outputs, filtering keys matching successful projections, then computes $P_0 - P_1$ to extract the exact baseline $\langle X_1 \rangle$ value.
   * **Duality Identity Check**: Computes the ratio between the LHS and RHS expectation values, evaluating how closely the numerical system satisfies the expected value of $0.25$.


In [3]:
"""
Implementation of the Kramers-Wannier Duality on a Quantum Computer.

This script verifies Equation (38) of our paper. It prepares
a random parameterised Ising state using a variational ansatz, applies a duality 
transformation D on it, and verifies the trace identity:

    Tr_H_hat( Z_1 * Z_2 * D(rho) ) = 1/4 * Tr_H( X_1 * P * rho )

The left-hand side (LHS) and right-hand side (RHS) expectations are computed by 
post-processing raw simulator results. Both quantum circuits evaluate the exact
same state instance to maintain verification identity conditions.

Author: Arif
"""

import numpy as np
from numpy import pi
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import TwoLocal
from qiskit.quantum_info import Operator, Pauli
from qiskit_aer import AerSimulator

# =====================================================================
# 1. Functional Circuit Component Generators
# =====================================================================

def build_ising_state_instruction(n, parameters):
    """Constructs a circuit for preparing a random initial state for ising."""
    circ_for_ising = QuantumCircuit(n)
    circ_for_ising.x(0)

    variational_form = TwoLocal(    # called variational form because it is typically used in Vartiational Quantum Eignesolver (VQE)
        n,
        rotation_blocks=["rz", "ry"],
        entanglement_blocks="cx",
        entanglement="linear",
        reps=1,
        insert_barriers=True,
    )
    ising_state = circ_for_ising.compose(variational_form)
    ising_state = ising_state.assign_parameters(parameters)

    # convert it to instruction
    return ising_state.to_instruction(label="ising state")


def build_duality_gate(n):
    """Constructs a gate to implement the duality transformation on the state."""
    ising = QuantumRegister(n, "ising")  # register holding ising qubits
    dwall = QuantumRegister(n, "dwall")  # register holding dual domain wall qubits 
    qc = QuantumCircuit(ising, dwall)

    # The operator curly C 
    qc.ryy(-pi/2, ising[n-1], dwall[n-1]) 
    qc.rxx(-pi/2, ising[n-1], dwall[n-1])

    for qbit in range(n-2, -1, -1):   # notice the start,stop,step in range due to quirky python conventions
        qc.swap(ising[qbit], dwall[qbit])

    # the operator curly S
    qc.rx(-pi/2, dwall[n-1])

    for qbit in range(n-2, -1, -1):  # again notice the quirky python range conventions
        qc.rzz(-pi/2, dwall[qbit], dwall[qbit+1])
        qc.rx(-pi/2, dwall[qbit])

    return qc.to_gate(label="CS")


def build_projection_gate(n):
    """Constructs a projection gate to be used for P and P^hat."""
    qr = QuantumRegister(n, "ising")  # register holding ising qubits
    ancilla = QuantumRegister(1, "ancilla")  # register holding ancilla
    qc = QuantumCircuit(qr, ancilla)   

    # projections on charge sectors (either ising or dwall)
    P_plus = 1/2 * (Operator(Pauli("I" * n)) + Operator(Pauli("X" * n)))
    P_minus = 1/2 * (Operator(Pauli("I" * n)) - Operator(Pauli("X" * n)))

    # operators to act on ancilla (of either ising or dwall)
    anc_id = Operator(Pauli("I"))
    anc_X = Operator(Pauli("X"))

    # define a tensored operator to act on lattice tensored ancillae, on ising and on dwall
    P = anc_id.tensor(P_plus) + anc_X.tensor(P_minus)
      
    qc.append(P, (qr[:] + ancilla[:]))
    return qc.to_gate(label="Proj")   

# =====================================================================
# 2. Execution Orchestration Routines
# =====================================================================

def execute_circuit_a(n, ising_state_instruct, duality_gate, projection_gate, backend, shots):
    """Constructs and executes Circuit A to extract duality expectation details."""
    ising = QuantumRegister(n, "ising")  
    dwall = QuantumRegister(n, "dwall")  
    ancilla_is = QuantumRegister(2, "ancilla_is")
    ancilla_dw = QuantumRegister(2, "ancilla_dw")
    anc_meas_reg = ClassicalRegister(4, "anc_meas_reg")  
    dw_meas_reg = ClassicalRegister(n, "dw_meas_reg")    

    qc = QuantumCircuit(ising, dwall, ancilla_is, ancilla_dw, anc_meas_reg, dw_meas_reg)

    qc.append(ising_state_instruct, ising[:])
    qc.barrier()
    qc.append(projection_gate, (ising[0:n] + ancilla_is[0:1]))
    qc.append(projection_gate, (dwall[0:n] + ancilla_dw[0:1]))
    qc.barrier()
    qc.append(duality_gate, (ising[0:n] + dwall[0:n]))
    qc.barrier()
    qc.append(projection_gate, (ising[0:n] + ancilla_is[1:2]))
    qc.append(projection_gate, (dwall[0:n] + ancilla_dw[1:2]))
    qc.barrier()

    # RESTORED: Absolute bit-by-bit index mappings from your verified original setup
    qc.measure(ancilla_is[0], anc_meas_reg[0])
    qc.measure(ancilla_dw[0], anc_meas_reg[1]) 
    qc.measure(ancilla_is[1], anc_meas_reg[2])
    qc.measure(ancilla_dw[1], anc_meas_reg[3]) 
    qc.measure(dwall[:], dw_meas_reg[:])

    job = backend.run(qc.decompose(reps=6), shots=shots, memory=False)
    return job.result().get_counts()


def execute_circuit_b(n, ising_state_instruct, projection_gate, backend, shots):
    """Constructs and executes Circuit B to extract baseline Ising reference details."""
    ising = QuantumRegister(n, "ising")  
    ancilla_is = QuantumRegister(1, "ancilla_is")
    qc_b = QuantumCircuit(ising, ancilla_is)

    qc_b.append(ising_state_instruct, ising[:])
    qc_b.barrier()
    qc_b.append(projection_gate, (ising[0:n] + [ancilla_is]))
    qc_b.barrier()
    qc_b.barrier()

    # Apply Hadamard ONLY on the zeroth ising qubit to enable X basis mapping
    qc_b.h(ising[0])  
    qc_b.measure_all()

    job_b = backend.run(qc_b.decompose(reps=6), shots=shots, memory=False)
    return job_b.result().get_counts()

# =====================================================================
# 3. Functional Post-Processing Calculators
# =====================================================================

def calculate_lhs_expectation(statistics, n, shots):
    """Processes Circuit A counts to calculate the Left-Hand Side expectation value."""
    dw_value = 0
    for key, value in statistics.items():
        if (key[n+1 : n+4+1] == '0000'):  # ancilla = 0000 corresponds to projections
            if (key[n-2 : n] == '00'):     # these are the first and second qubit of dwall
                dw_value += value  
            elif (key[n-2 : n] == '01'):
                dw_value -= value
            elif (key[n-2 : n] == '10'):
                dw_value -= value
            elif (key[n-2 : n] == '11'):
                dw_value += value   
    return dw_value / shots


def calculate_rhs_expectation(statistics_b, n, shots):
    """Processes Circuit B counts to calculate the Right-Hand Side expectation value."""
    P_0 = 0  
    P_1 = 0  
    for key, value in statistics_b.items():
        if (key[0] == '0'):      # ancilla = 0 ensures the projection operator 
            if (key[n] == '0'):    
                 P_0 += value / shots
            elif (key[n] == '1'):
                 P_1 += value / shots
    return P_0 - P_1

# =====================================================================
# 4. Main Runtime Pipeline Execution
# =====================================================================

def main():
    # User Constants Definition
    NUM_LATTICE_SITES = 6
    NUM_SHOTS = 2**16

    # Aliasing parameters cleanly to track internal sizing math
    n = NUM_LATTICE_SITES
    shots = NUM_SHOTS

    # Setup the simulator backend pipeline instance
    backend = AerSimulator()

    # Generate the shared parameter vector globally to guarantee state identity
    parameters = (2 * np.pi * np.random.rand(4 * n))

    # Step A: Build identical state blueprint instruction to feed into both systems
    ising_state_instruct = build_ising_state_instruction(n, parameters)
    
    # Step B: Compile auxiliary matrix operators 
    duality_gate = build_duality_gate(n)
    projection_gate = build_projection_gate(n)

    # Step C: Run Executions using the exact same random state asset instance
    statistics_a = execute_circuit_a(n, ising_state_instruct, duality_gate, projection_gate, backend, shots)
    statistics_b = execute_circuit_b(n, ising_state_instruct, projection_gate, backend, shots)

    # Step D: Evaluate Expectations
    dw_exp = calculate_lhs_expectation(statistics_a, n, shots)
    ising_exp = calculate_rhs_expectation(statistics_b, n, shots)

    # Step E: Print Identity Report Outputs Summary
    ratio = dw_exp / ising_exp if ising_exp != 0 else 0
    
    print(f"==================================================")
    print(f"   Kramers-Wannier Quantum Duality Report (n={n})  ")
    print(f"==================================================")
    print(f"LHS Trace Expectation (dw_exp)    : {dw_exp:.6f}")
    print(f"RHS Baseline Expectation (ising_exp): {ising_exp:.6f}")
    print(f"Final Verification Ratio (LHS/RHS): {ratio:.6f} (Expected: ~0.25)")
    print(f"==================================================")

if __name__ == "__main__":
    main()


   Kramers-Wannier Quantum Duality Report (n=6)  
LHS Trace Expectation (dw_exp)    : -0.048431
RHS Baseline Expectation (ising_exp): -0.195679
Final Verification Ratio (LHS/RHS): 0.247505 (Expected: ~0.25)
